Cell 1: Load Dataset

In [ ]:
import os
import pandas as pd
import numpy as np

"""
    Read .csv files and store data into an array
    format: |LOS|NLOS|data...|
"""
rootdir = '../dataset/UWB-LOS-NLOS-Data-Set/dataset/'
df = pd.DataFrame()
first = 1
for dirpath, _, filenames in os.walk(rootdir):
    for file in filenames:
        filename = os.path.join(dirpath, file)
        print(filename)
        # read data from file
        oneFileDF = pd.read_csv(filename)
        # append to array
        if first > 0:
            first = 0
            df = oneFileDF
        else:
            df = pd.concat([df, oneFileDF], ignore_index=True)

df

Cell 2: Normalize CIR Data by RXPACC

In [ ]:
cir_columns = [f'CIR{i}' for i in range(1016)]
df[cir_columns] = df[cir_columns].div(df['RXPACC'], axis=0)

df[cir_columns].head()


These CIR values are normalized by RXPACC, meaning each original CIR amplitude is divided by the number of received preamble symbols (RXPACC). This normalization ensures each CIR value represents the average amplitude per preamble symbol received, rather than the total accumulated energy.

Why Normalize by RXPACC?

It standardizes the CIR data, removing variations caused by differing amounts of received preamble symbols.
Ensures each CIR sample reflects consistent measurement conditions across all samples.

Layman Analogy for Dividing CIR by RXPACC (CIR Signal Perspective)
Imagine you're clapping your hands in a large empty room, and a microphone is recording the sound.

If you clap once, the microphone picks up a single echo.
If you clap 10 times, the microphone records more total sound, but that doesn’t mean each clap was louder—there were just more claps.
Now, if you want to measure how strong each individual clap’s echo is, you need to divide the total recorded sound by the number of claps.

🔹 In CIR measurements, the CIR values represent the total signal received, and RXPACC is like the number of claps (received preamble symbols).
🔹 By dividing CIR by RXPACC, we measure the true signal strength per preamble symbol, rather than just the total accumulated signal.

🚀 This ensures fair comparisons between different CIR measurements, even if they have different numbers of preamble symbols!

Cell 3: Perform PCA on CIR Data (linearity check)

In [ ]:
from sklearn.decomposition import PCA

cir_df = df[cir_columns]

pca = PCA(n_components=20)
pca.fit(cir_df)

explained_variance_ratio = pca.explained_variance_ratio_
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print('Explained variance per component:', explained_variance_ratio)
print('Cumulative explained variance:', cumulative_explained_variance)


Cell 4: PCA Explained Variance Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range(1, 21), cumulative_explained_variance, marker='o', linestyle='--', color='blue')
plt.xlabel('Number of PCA Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance by PCA Components (CIR Features)')
plt.xticks(range(1, 21))
plt.grid(True)
plt.show()


This PCA plot clearly shows that about 94% of the variance is captured by the first 20 PCA components. This means the CIR data is linear enough, making PCA an effective choice to reduce dimensions while keeping most of the dataset's important details.

Cell 5: Scatter Plot (Pairplot) for Visual Linearity Check (Subset of CIR Features)

In [ ]:
import seaborn as sns

# select 5 CIR features for clear visualization
sample_df = df[['CIR0', 'CIR2', 'CIR4', 'CIR6', 'CIR800']].sample(500, random_state=42)

sns.pairplot(sample_df, kind='reg',
             plot_kws={'scatter_kws': {'alpha':0.3}, 'line_kws': {'color':'red'}})
plt.suptitle('Scatter Plot - Visual Linearity Check on CIR', y=1.02)
plt.show()


This scatter plot clearly indicates that the selected CIR features have a moderately linear relationship, as most points align reasonably well around the linear regression lines (in red). While not perfectly linear due to some scatter, the trend is clear enough, confirming that PCA is suitable for dimensionality reduction on these CIR features.

In [ ]:
from sklearn.decomposition import PCA

cir_df = df[cir_columns]

# Use PCA (e.g., 10 components)
pca = PCA(n_components=10)
cir_pca = pca.fit_transform(cir_df)


In [ ]:
print("Explained Variance per component:", pca.explained_variance_ratio_)
print("Total Explained Variance:", pca.explained_variance_ratio_.sum())


In [ ]:
pca_columns = [f'CIR_PCA{i+1}' for i in range(cir_pca.shape[1])]
cir_pca_df = pd.DataFrame(cir_pca, columns=pca_columns)

# Quickly verify results
cir_pca_df.head()


In [ ]:
final_df = pd.concat([df[['NLOS', 'RANGE', 'FP_AMP1', 'FP_AMP2', 'FP_AMP3', 'CIR_PWR', 'MAX_NOISE']], cir_pca_df], axis=1)

# Check final dataset
final_df.head()
